In [0]:
# Databricks notebook source

# COMMAND ----------
# Formula 1 Racing Data Platform
# Notebook: 02_data_profiling
#
# Purpose:
# Analyse Bronze tables and generate a data profiling report.

In [0]:
%run ../common/00_configure_catalog

In [0]:
# COMMAND ----------
# Imports

from pyspark.sql.functions import col, when, count

In [0]:
# COMMAND ----------
# Get Bronze Tables

tables = (
    spark.sql(f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}")
)

display(tables)

In [0]:
# COMMAND ----------
# Data Profiling

profiling_results = []
column_results = []
dictionary = []

for row in tables.collect():

    table_name = row.tableName

    print("=" * 80)
    print(f"Profiling table: {table_name}")
    print("=" * 80)

    df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}")

    # Basic Metrics
    row_count = df.count()
    column_count = len(df.columns)

    print(f"Rows    : {row_count}")
    print(f"Columns : {column_count}")

    print("\nSchema")
    df.printSchema()

    print("\nSummary Statistics")
    display(df.describe())

    # Duplicate rows
    duplicate_rows = row_count - df.dropDuplicates().count()

    # Null values
    null_counts = (
        df.select([
            count(when(col(c).isNull(), c)).alias(c)
            for c in df.columns
        ])
    )

    print("\nNull Values")
    display(null_counts)

    profiling_results.append({
        "table_name": table_name,
        "rows": row_count,
        "columns": column_count,
        "duplicate_rows": duplicate_rows
    })

    # Column profiling
    for field in df.schema.fields:

        column_name = field.name
        print(f"\n 🔎 Analysing column {column_name}...")
        data_type = field.dataType.simpleString()
        nullable = field.nullable

        null_count = (
            df.filter(col(column_name).isNull()).count()
        )

        null_pct = round(
            (null_count / row_count) * 100,
            2
        ) if row_count > 0 else 0

        distinct_count = (
            df.select(column_name)
            .distinct()
            .count()
        )

        is_candidate_key = (
        distinct_count == total_rows and
        null_count == 0
        )

        column_results.append({
            "table_name": table_name,
            "column_name": column_name,
            "data_type": data_type,
            "nullable": nullable,
            "null_count": null_count,
            "null_percentage": null_pct,
            "distinct_values": distinct_count,
            "is_candidate_key": is_candidate_key
        })

        # Data Dictionary
        dictionary.append({
        "table_name": table_name,
        "column_name": column_name,
        "data_type": data_type,
        "nullable": nullable,
        "description": ""
        })

In [0]:
# COMMAND ----------
# Profiling Summary

summary_df = spark.createDataFrame(profiling_results)

display(summary_df)

In [0]:
# COMMAND ----------
# Profiling Columns
profiling_columns_df = spark.createDataFrame(column_results)

display(profiling_columns_df)

In [0]:
# COMMAND ----------
# Dictionary
dictionary_df = spark.createDataFrame(dictionary)

display(dictionary_df)

In [0]:
# COMMAND ----------
# Save Profiling Summary Report

(
    summary_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{METADATA_SCHEMA}.profiling_summary")
)

In [0]:
# COMMAND ----------
# Save Profiling Column Report

(
    profiling_columns_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{METADATA_SCHEMA}.profiling_columns")
)

In [0]:
# COMMAND ----------
# Save Dictionary Report

(
    dictionary_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{METADATA_SCHEMA}.data_dictionary")
)

In [0]:
# COMMAND ----------
# Validation

display(
    spark.table(f"{CATALOG}.{METADATA_SCHEMA}.profiling_summary")
)

In [0]:
# COMMAND ----------
# Validation

display(
    spark.table(f"{CATALOG}.{METADATA_SCHEMA}.profiling_columns")
)

In [0]:
# COMMAND ----------
# Validation

display(
    spark.table(f"{CATALOG}.{METADATA_SCHEMA}.data_dictionary")
)